### Top 3 highest-grossing sales reps for each store.

https://www.databricks.com/blog/2015/07/15/introducing-window-functions-in-spark-sql.html

``` dense_rank() OVER (PARTITION BY category ORDER BY revenue DESC) as rank ```

In [1]:
from pyspark.sql import SparkSession
import getpass

username = getpass.getuser()

In [2]:
spark = SparkSession.builder \
.config("spark.port.ui", 0) \
.config("spark.sql.warehouse.dir", f"/user/{username}/warehouse") \
.enableHiveSupport() \
.master("yarn") \
.getOrCreate()

In [3]:
from pyspark.sql import Window
import pyspark.sql.functions as F

# 1. Generate 50,000 rows of synthetic sales data
sales_df = spark.range(50000) \
    .withColumn("store_id", (F.rand() * 50).cast("int")) \
    .withColumn("sales_rep_id", (F.rand() * 500).cast("int")) \
    .withColumn("revenue", (F.rand() * 10000).cast("double"))

In [10]:
sales_df.filter("sales_rep_id = 201 and store_id = 0")

id,store_id,sales_rep_id,revenue
7026,0,201,9995.762927478841
34281,0,201,8940.096377805614


In [5]:
# 2. Define the Window
# Partition by store (reset rank per store), Order by revenue descending
window_spec = Window.partitionBy("store_id").orderBy(F.col("revenue").desc())

In [6]:
# 3. Apply the Function
ranked_sales = sales_df.withColumn("rank", F.dense_rank().over(window_spec))

In [7]:
# 4. Filter for Top 3
top_reps = ranked_sales.filter(F.col("rank") <= 3)
top_reps.orderBy("store_id", "rank").show()

+-----+--------+------------+-----------------+----+
|   id|store_id|sales_rep_id|          revenue|rank|
+-----+--------+------------+-----------------+----+
| 7026|       0|         201|9995.762927478841|   1|
|15400|       0|         211| 9992.68479308532|   2|
|19853|       0|          78|9989.322444588724|   3|
|33994|       1|         433|9997.147054182853|   1|
|29689|       1|          47|9988.859328719589|   2|
|11714|       1|         317| 9973.90725520129|   3|
|21999|       2|         399|9996.413170591799|   1|
|27871|       2|         403| 9989.61144226108|   2|
|19556|       2|         150| 9988.79421903533|   3|
|40138|       3|         173|9975.888236392146|   1|
|28672|       3|         122|9947.648478590047|   2|
| 6280|       3|         478| 9947.41653197401|   3|
| 2941|       4|         260|9997.795293776317|   1|
|39682|       4|         130|9988.932013705873|   2|
| 9975|       4|         167|9986.969841553217|   3|
|11084|       5|         387|9988.085199693664

In [8]:
# Just playing around
row_sales = sales_df.withColumn("rank", F.row_number().over(window_spec))

In [9]:
row_sales.show()

+-----+--------+------------+-----------------+----+
|   id|store_id|sales_rep_id|          revenue|rank|
+-----+--------+------------+-----------------+----+
| 9495|      31|         154|9999.171189993804|   1|
|27776|      31|         145|9983.985595434327|   2|
|19146|      31|         339|9977.390545359203|   3|
|10600|      31|         356| 9968.95565404774|   4|
|40210|      31|         419|9946.159414700036|   5|
|49045|      31|         155| 9938.69808883279|   6|
| 8451|      31|         283|9933.113976146888|   7|
|29102|      31|          78| 9931.10391268702|   8|
|40886|      31|         396| 9920.58681673777|   9|
|23854|      31|         438|9915.437781832172|  10|
|36718|      31|         278|9900.151476456673|  11|
|23449|      31|         141|9870.875413200778|  12|
| 3563|      31|         380|9867.414297593308|  13|
|37850|      31|         168| 9860.96455905614|  14|
| 1093|      31|         345|9860.424966241606|  15|
| 7943|      31|         473|9844.014083016633